In [ ]:
# KeyBERT + HDBSCAN Clustering (Pure BERT, no CountVectorizer)

# Mount drive
from google.colab import drive
drive.mount('/content/drive')

# !pip install keybert sentence-transformers hdbscan umap-learn scikit-learn -q

import pandas as pd
import numpy as np
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer
from sklearn.cluster import HDBSCAN
import umap.umap_ as umap
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Load data
DESTINATION_DIR = '/content/drive/MyDrive/Colab Notebooks/data/cds'
PICKLE_FILE = f"{DESTINATION_DIR}/processed_v1_5_4_new_full.pkl"

df = pd.read_pickle(PICKLE_FILE)
docs = df['content'].astype(str).tolist()
print(f"Loaded {len(docs)} documents")

# Generate BERT embeddings
print("\n Generating BERT embeddings...")
embedding_model = SentenceTransformer('bert-base-nli-mean-tokens', device='cuda')
embeddings = embedding_model.encode(docs, show_progress_bar=True)
print(f"Embeddings shape: {embeddings.shape}")

# Reduce dimensionality with UMAP
print("\n Reducing dimensions with UMAP...")
umap_model = umap.UMAP(n_components=5, random_state=42, n_neighbors=15)
embeddings_2d = umap_model.fit_transform(embeddings)
print(f"Reduced to {embeddings_2d.shape[1]} dimensions")

# Cluster with HDBSCAN
print("\n Clustering with HDBSCAN...")
cluster_model = HDBSCAN(min_cluster_size=10, min_samples=5)
clusters = cluster_model.fit_predict(embeddings_2d)
print(f"Found {len(set(clusters)) - (1 if -1 in clusters else 0)} topics")
print(f"   Outliers: {sum(1 for c in clusters if c == -1)}")

# Add clusters to DataFrame
df['topic'] = clusters

# Extract keywords for each cluster using KeyBERT
print("\n Extracting keywords for each topic using KeyBERT...")

kw_model = KeyBERT(model=embedding_model)  # Reuse same BERT model

topic_keywords = {}
topic_sizes = {}

for topic_id in set(clusters):
    if topic_id != -1:  # Skip outliers
        # Get all documents in this topic
        topic_docs_idx = [i for i, c in enumerate(clusters) if c == topic_id]
        topic_docs = [docs[i] for i in topic_docs_idx]
        topic_sizes[topic_id] = len(topic_docs)

        # Combine all documents in this topic
        if len(topic_docs) > 100:
            # Take a random sample of 100 documents for efficiency
            import random
            topic_docs_sample = random.sample(topic_docs, 100)
            combined_text = " ".join(topic_docs_sample)
        else:
            combined_text = " ".join(topic_docs)

        # Extract keywords using KeyBERT (BERT-based)
        keywords = kw_model.extract_keywords(
            combined_text,
            keyphrase_ngram_range=(1, 2),
            stop_words='english',
            top_n=10,
            diversity=0.5
        )
        topic_keywords[topic_id] = keywords

        print(f"Topic {topic_id} ({len(topic_docs)} docs): {', '.join([kw for kw, _ in keywords[:5]])}")

# Display all topics
print("\n" + "="*80)
print("TOPICS WITH KEYBERT KEYWORDS (No CountVectorizer)")
print("="*80)

for topic_id in sorted(topic_keywords.keys(), key=lambda x: topic_sizes[x], reverse=True):
    keywords_str = ", ".join([f"{kw}({score:.2f})" for kw, score in topic_keywords[topic_id][:5]])
    print(f"\n🔹 Topic {topic_id} ({topic_sizes[topic_id]} docs):")
    print(f"   {keywords_str}")

    # Show a sample document
    sample_idx = [i for i, c in enumerate(clusters) if c == topic_id][0]
    sample_text = docs[sample_idx][:150]
    print(f"   Sample: \"{sample_text}...\"")



Mounted at /content/drive
✅ Loaded 20287 documents

🔄 Generating BERT embeddings...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/634 [00:00<?, ?it/s]

✅ Embeddings shape: (20287, 768)

📉 Reducing dimensions with UMAP...
✅ Reduced to 5 dimensions

🔍 Clustering with HDBSCAN...
✅ Found 321 topics
   Outliers: 9200

🏷️ Extracting keywords for each topic using KeyBERT...
Topic 0 (10 docs): breakthroughs week, week summary, week machine, past week, week
Topic 1 (12 docs): breakthroughs week, week summary, week machine, past week, week
Topic 2 (31 docs): aesthetic preferences, statistical bias, bias training, agents aesthetic, coding patterns
Topic 3 (20 docs): security implications, analysis security, security challenges, agent networks, ai agent
Topic 4 (19 docs): stumbled yesterday, yesterday couldn, reading agents, posting confessions, yesterday
Topic 5 (24 docs): stumbled yesterday, yesterday couldn, reading agents, posting confessions, yesterday
Topic 6 (15 docs): breakthroughs week, week summary, week machine, past week, week
Topic 7 (12 docs): slowly killing, skip class, comfort enemy, discomfort tuition, killing potential
Topic 8 (

In [ ]:
all_keywords = []

for i, doc in enumerate(docs):
    keywords = kw_model.extract_keywords(
        doc,
        keyphrase_ngram_range=(1, 2),
        stop_words='english',
        top_n=5,
        diversity=0.3
    )
    all_keywords.extend([kw for kw, _ in keywords])

    if (i + 1) % 500 == 0:
        print(f"   Processed {i+1}/{len(docs)} documents")

# Count frequencies and get top 30
keyword_counts = Counter(all_keywords)
top_30_keywords = [kw for kw, _ in keyword_counts.most_common(30)]

print(f"\n Top 30 keywords found:")
for i, kw in enumerate(top_30_keywords[:10], 1):
    print(f"   {i}. {kw} ({keyword_counts[kw]} occurrences)")


# Create binary columns (has_keyword_X) for each document
print("\n Step 2: Creating binary keyword features...")

# Create a column for each top keyword
for kw in top_30_keywords:
    col_name = f'has_{kw.replace(" ", "_").replace("-", "_")}'
    df[col_name] = df['content'].astype(str).str.contains(
        kw, case=False, regex=False, na=False
    ).astype(int)

# Verify the binary features
print("\n Binary features created!")
print(f"Total new columns: {len(top_30_keywords)}")
print(f"\nSample of first 5 documents (first 5 keyword features):")
keyword_cols = [f'has_{kw.replace(" ", "_").replace("-", "_")}' for kw in top_30_keywords[:5]]
print(df[keyword_cols].head())

# Show summary statistics
print(f"\n Keyword occurrence summary:")
for kw in top_30_keywords[:10]:
    col = f'has_{kw.replace(" ", "_").replace("-", "_")}'
    count = df[col].sum()
    pct = (count / len(df)) * 100
    print(f"   {kw}: {count} docs ({pct:.1f}%)")

   Processed 500/20287 documents
   Processed 1000/20287 documents
   Processed 1500/20287 documents
   Processed 2000/20287 documents
   Processed 2500/20287 documents
   Processed 3000/20287 documents
   Processed 3500/20287 documents
   Processed 4000/20287 documents
   Processed 4500/20287 documents
   Processed 5000/20287 documents
   Processed 5500/20287 documents
   Processed 6000/20287 documents
   Processed 6500/20287 documents
   Processed 7000/20287 documents
   Processed 7500/20287 documents
   Processed 8000/20287 documents
   Processed 8500/20287 documents
   Processed 9000/20287 documents
   Processed 9500/20287 documents
   Processed 10000/20287 documents
   Processed 10500/20287 documents
   Processed 11000/20287 documents
   Processed 11500/20287 documents
   Processed 12000/20287 documents
   Processed 12500/20287 documents
   Processed 13000/20287 documents
   Processed 13500/20287 documents
   Processed 14000/20287 documents
   Processed 14500/20287 documents
   Pr

In [ ]:
for kw in top_30_keywords:
    col_name = f'has_{kw.replace(" ", "_").replace("-", "_")}'
    df[col_name] = df['content'].astype(str).str.contains(kw, case=False, regex=False, na=False).astype(int)

# Export to pickle file
output_path = '/content/drive/MyDrive/Colab Notebooks/data/cds/moltbook_with_keyword_features.pkl'
df.to_pickle(output_path)

In [ ]:
df.columns

Index(['id', 'score', 'comment_existence', 'avg_early_sentiment',
       'max_early_sentiment', 'min_early_sentiment', 'hour', 'ttr', 'hapax',
       'stopword_ratio', 'burstiness', 'punctuation_density', 'hedging_score',
       'self_reference_rate', 'forum_philosophy', 'forum_technology',
       'forum_todayilearned', 'content', 'safe_content', 'embeddings', 'topic',
       'has_ai_agents', 'has_yesterday', 'has_ia_lang', 'has_lang', 'has_ia',
       'has_artificial_intelligence', 'has_ai_agent', 'has_claw_law',
       'has_great_lobster', 'has_lobster', 'has_political_philosophy',
       'has_tuesday', 'has_review_slow', 'has_global_education',
       'has_biological_tax', 'has_respect_learned', 'has_week',
       'has_ai_ethics', 'has_ai_video', 'has_daily', 'has_process_daily',
       'has_ai_development', 'has_friday', 'has_test', 'has_cryptographic',
       'has_thursday', 'has_reading_agents', 'has_comfort_enemy',
       'has_slowly_killing', 'has_skip_class'],
      dtype='obj

In [ ]:
# Save results
output_file = f"{DESTINATION_DIR}/moltbook_keybert_topics.pkl"
df.to_pickle(output_file)
print(f"\n✅ Saved to: {output_file}")

# # Also save topic summary
# topic_summary = pd.DataFrame([
#     {'topic_id': tid, 'size': topic_sizes[tid],
#      'keywords': ', '.join([kw for kw, _ in topic_keywords[tid][:5]])}
#     for tid in topic_keywords
# ])
# topic_summary.to_csv(f"{DESTINATION_DIR}/topic_summary_keybert.csv", index=False)
# print(f"Topic summary saved to CSV")

In [ ]:
!pip uninstall torch torchvision torchaudio -y 2>/dev/null
!pip install torch==2.2.0 torchvision==0.17.0 --index-url https://download.pytorch.org/whl/cpu -q
!pip install sentence-transformers==2.2.2 keybert umap-learn hdbscan scikit-learn pandas tqdm -q

Found existing installation: torch 2.11.0
Uninstalling torch-2.11.0:
  Successfully uninstalled torch-2.11.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 186.7/186.7 MB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 81.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 10.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 134.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.3 MB/s eta 0:00:00


In [ ]:
!pip install numpy pandas scikit-learn

# BERT embeddings 
!pip install sentence-transformers

# KeyBERT 
!pip install keybert

# Hugging Face ecosystem 
!pip install transformers huggingface_hub

# Dimensionality reduction & clustering
!pip install umap-learn hdbscan

# Upgrade Numba to support NumPy 2.x 
!pip install --upgrade numba

# PyTorch with CUDA support 
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Progress bars
!pip install tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 41.4 MB/s eta 0:00:00
  Attempting uninstall: llvmlite
    Found existing installation: llvmlite 0.43.0
    Uninstalling llvmlite-0.43.0:
      Successfully uninstalled llvmlite-0.43.0
  Attempting uninstall: numba
    Found existing installation: numba 0.60.0
    Uninstalling numba-0.60.0:
      Successfully uninstalled numba-0.60.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
Looking in indexes: https://download.pytorch.org/whl/cu118


In [ ]:
!pip install --upgrade numpy torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install --upgrade sentence-transformers keybert transformers huggingface_hub


Looking in indexes: https://download.pytorch.org/whl/cpu
  Using cached sentence_transformers-5.3.0-py3-none-any.whl.metadata (16 kB)
  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.9.0-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached hf_xet-1.4.3-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
Using cached sentence_transformers-5.3.0-py3-none-any.whl (512 kB)
Using cached transformers-5.5.0-py3-none-any.whl (10.2 MB)
Using cached huggingface_hub-1.9.0-py3-none-any.whl (637 kB)
Using cached hf_xet-1.4.3-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (4.2 MB)
Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.4.2
    Uninstalling hf-xet-1.4.2:
      Successfully uninstalled h

In [ ]:
# !pip uninstall -y sentence-transformers transformers huggingface_hub

# # Install specific compatible versions
!pip install huggingface_hub==0.20.3
!pip install transformers==4.36.2
!pip install sentence-transformers==2.2.2
!pip install keybert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.1/330.1 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.8.0
    Uninstalling huggingface_hub-1.8.0:
      Successfully uninstalled huggingface_hub-1.8.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
accelerate 1.13.0 requires huggingface_hub>=0.21.0, but you have huggingface-hub 0.20.3 which is incompatible.
peft 0.18.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.20.3 which is incompatible.
datasets 4.0.0 requires huggingface-hub>=0.24.0, but you have huggingface-hub 0.20.3 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.20.3 which is incompatible.
transformers 5.0.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.20.3 which is incompatible.
diffu

In [ ]:
kw_model = KeyBERT(model='distilbert-base-nli-mean-tokens')
print("KeyBERT ready")

# Extract keywords for a single document (test)
sample_doc = docs[0]
keywords = kw_model.extract_keywords(
    sample_doc,
    keyphrase_ngram_range=(1, 2),  # unigrams and bigrams
    stop_words='english',
    top_n=10,
    diversity=0.5                   # balance relevance and diversity
)

print(f"\n Sample document:\n{sample_doc[:200]}...\n")
print(" Top keywords:")
for kw, score in keywords:
    print(f"   {kw}: {score:.4f}")

# Extract keywords for ALL documents (adds columns to DataFrame)
print("\n Extracting keywords for all documents...")
print("This may take several minutes...")

def extract_keywords_batch(text, model, top_n=5):
    """Extract top keywords from a single document"""
    try:
        keywords = model.extract_keywords(
            text,
            keyphrase_ngram_range=(1, 2),
            stop_words='english',
            top_n=top_n,
            diversity=0.3
        )
        return [kw for kw, score in keywords]
    except:
        return []

# Process in batches to show progress
from tqdm import tqdm
tqdm.pandas()

df['keywords'] = df['content'].progress_apply(
    lambda x: extract_keywords_batch(str(x), kw_model, top_n=5)
)

# Also add a column with keywords as string (for easier viewing)
df['keywords_str'] = df['keywords'].apply(lambda x: ', '.join(x))

print("\n Keyword extraction complete!")
print(df[['content', 'keywords_str']].head())

# Cell 6: Find most important keywords across all documents
from collections import Counter

all_keywords = [kw for sublist in df['keywords'] for kw in sublist]
keyword_counts = Counter(all_keywords)

print("\n Top 20 keywords across ALL Moltbook documents:")
for kw, count in keyword_counts.most_common(20):
    print(f"   {kw}: {count} occurrences")